In [31]:
import pandas as pd

EVAL_PATH = "Data/eval/norskgpt-llama3-8b-apptainer-checkpoint-5000-inputs-refs-preds-examples_1000.jsonl"
df = pd.read_json(EVAL_PATH, lines=True)
df.shape
df.head(1)
df.iloc[0].reference+ "\n\n"

'Kinn kommune har godkjent søknaden fra Fjord Base AS for et tilbygg til eksisterende næringsbygg på Botnaneset 27. Tilbygget, som er på 42,67 m², vil huse et kontrollrom og kontorarbeidsplass, og tiltaket har blitt vurdert som i samsvar med gjeldende reguleringer og uten miljøfare. Vedtaket kan påklages innen tre uker.\n\n'

In [32]:
EVAL_PATH = "Data/eval/gemma-2-9b-checkpoint-5000-inputs-refs-preds-examples_1000.jsonl"
df = pd.read_json(EVAL_PATH, lines=True)
df.shape
df.head(1)
df.iloc[0].reference+ "\n\n"

'Kinn kommune har godkjent søknaden fra Fjord Base AS for et tilbygg til eksisterende næringsbygg på Botnaneset 27. Tilbygget, som er på 42,67 m², vil huse et kontrollrom og kontorarbeidsplass, og tiltaket har blitt vurdert som i samsvar med gjeldende reguleringer og uten miljøfare. Vedtaket kan påklages innen tre uker.\n\n'

'Kinn kommune har godkjent søknaden fra Fjord Base AS for et tilbygg til eksisterende næringsbygg på Botnaneset 27. Tilbygget, som er på 42,67 m², vil huse et kontrollrom og kontorarbeidsplass, og tiltaket har blitt vurdert som i samsvar med gjeldende reguleringer og uten miljøfare. Vedtaket kan påklages innen tre uker.\n\n'

In [28]:
EVAL_DIR = "Data/eval"
RUN_SUFFIX = "-checkpoint-5000-inputs-refs-preds-examples_1000.jsonl"

MODEL_RUNS = [
    "eurollm-9b-instruct-apptainer",
    "gemma-2-27b-apptainer",
    "gemma-2-9b-apptainer",
    "gemma-2b-apptainer",
    "gemma-3-12b-apptainer",
    "gemma-7b-apptainer",
    "llama-2-13b-chat-norwegian-apptainer",
    "llama-3.1-8b-instruct-apptainer",
    "nb-gpt-j-6b-apptainer",
    "normistral-11b-apptainer",
    "normistral-7b-apptainer",
    "normistral-7b-instruct-apptainer",
    "norskgpt-llama3-8b-apptainer",
    "norwai-mistral-7b-instruct-apptainer",
    "viking-13b-apptainer",
    "viking-33b-apptainer",
    "viking-7b-apptainer",
]

for run_name in MODEL_RUNS:
    run_path = f"{EVAL_DIR}/{run_name}{RUN_SUFFIX}"
    run_df = pd.read_json(run_path, lines=True)
    print(run_df.iloc[100].input_text[:30] + "\n\n")

/var/folders/jl/td84cp9172vgb3dwjmyznhvr0000gn/T/ipykernel_3666/3436246497.py:26: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  run_df = pd.read_json(run_path, lines=True)


ValueError: Expected object or value

In [105]:
# DeepEval G-Eval — run after loading `df`. Tune SAMPLE_IDX.
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple

import nest_asyncio
import pandas as pd
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

nest_asyncio.apply()

CLIP_MAX = 60_000
SAMPLE_IDX = 0

CORRECTNESS_CRITERIA = (
    "Vurder om den genererte oppsummeringen er korrekt i forhold til kildedokumentet (kontekst) "
    "og samsvarer godt med referanseoppsummeringen der det er relevant."
)


def clip_text(s, max_chars: int = CLIP_MAX) -> str:
    s = str(s)
    return s if len(s) <= max_chars else s[:max_chars] + "\n[... truncated ...]"


def summarization_test_case(
    df: pd.DataFrame, row_idx: int
) -> Tuple[LLMTestCase, List[LLMTestCaseParams]]:
    row = df.iloc[row_idx]
    ctx = [clip_text(row["input_text"])] if pd.notna(row.get("input_text")) else None
    eval_params = [
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ]
    if ctx is not None:
        eval_params.append(LLMTestCaseParams.CONTEXT)
    tc = LLMTestCase(
        input=clip_text(row["prompt"]),
        actual_output=str(row["prediction"]),
        expected_output=str(row["reference"]),
        context=ctx,
    )
    return tc, eval_params


def correctness_geval(
    evaluation_params: List[LLMTestCaseParams],
    model: Optional[object] = None,
    *,
    verbose: bool = True,
) -> GEval:
    kwargs = dict(
        name="Correctness",
        criteria=CORRECTNESS_CRITERIA,
        evaluation_params=evaluation_params,
        verbose_mode=verbose,
    )
    if model is not None:
        kwargs["model"] = model
    return GEval(**kwargs)


def run_correctness(
    judge_name: str,
    df: pd.DataFrame,
    row_idx: int,
    model: Optional[object],
    *,
    verbose: bool = True,
) -> Tuple[float, GEval]:
    """model=None → DeepEval default OpenAI judge (needs OPENAI_API_KEY)."""
    test_case, eval_params = summarization_test_case(df, row_idx)
    metric = correctness_geval(evaluation_params=eval_params, model=model, verbose=verbose)
    score = metric.measure(test_case)
    print(f"{judge_name} — score (0–1): {score}\nReason:\n{metric.reason}")
    return score, metric


@dataclass(frozen=True)
class JudgeSpec:
    label: str
    env_var: Optional[str]
    factory: Callable[[], Optional[object]]


def _judge_openai() -> None:
    return None


def _judge_claude_haiku():
    from deepeval.models import AnthropicModel

    return AnthropicModel(model="claude-3-5-haiku-20241022")


def _judge_gemini_flash():
    from deepeval.models import GeminiModel

    return GeminiModel(model="gemini-2.5-flash")


def _judge_mistral_medium():
    try:
        import litellm  # noqa: F401
    except ImportError as e:
        raise ImportError("pip install litellm") from e
    from deepeval.models import LiteLLMModel

    return LiteLLMModel(
        model="mistral/mistral-medium-latest",
        api_key=os.environ["MISTRAL_API_KEY"],
    )


JUDGES: Dict[str, JudgeSpec] = {
    "openai": JudgeSpec("OpenAI (default)", "OPENAI_API_KEY", _judge_openai),
    "claude_haiku": JudgeSpec("Claude 3.5 Haiku", "ANTHROPIC_API_KEY", _judge_claude_haiku),
    "gemini_flash": JudgeSpec("Gemini 2.5 Flash", "GOOGLE_API_KEY", _judge_gemini_flash),
    "mistral_medium": JudgeSpec("Mistral Medium", "MISTRAL_API_KEY", _judge_mistral_medium),
}


def list_judge_ids() -> List[str]:
    return list(JUDGES.keys())


def run_judge(
    judge_id: str,
    df: pd.DataFrame,
    row_idx: int = SAMPLE_IDX,
    *,
    verbose: bool = True,
) -> Tuple[float, GEval]:
    """One G-Eval call for a registered judge id (see `list_judge_ids()`)."""
    if judge_id not in JUDGES:
        raise KeyError(f"Unknown judge {judge_id!r}. Use one of: {list(JUDGES.keys())}")
    spec = JUDGES[judge_id]
    if spec.env_var and not os.environ.get(spec.env_var):
        raise RuntimeError(f"Set {spec.env_var} before running judge {judge_id!r}.")
    model = spec.factory()
    return run_correctness(spec.label, df, row_idx, model, verbose=verbose)


def run_judges(
    judge_ids: List[str],
    df: pd.DataFrame,
    row_idx: int = SAMPLE_IDX,
    *,
    verbose: bool = True,
) -> Dict[str, float]:
    """Run several judges in sequence (one API bill per judge)."""
    scores: Dict[str, float] = {}
    for jid in judge_ids:
        score, _ = run_judge(jid, df, row_idx, verbose=verbose)
        scores[jid] = score
    return scores

In [106]:
# Registered judges — set the matching API key, then:
#   run_judge("openai", df)              # or claude_haiku | gemini_flash | mistral_medium
#   run_judge("claude_haiku", df, row_idx=5)
#   run_judges(["openai", "gemini_flash"], df)   # runs in order; one charge per judge
# Custom model (not in JUDGES): run_correctness("My judge", df, SAMPLE_IDX, my_deepeval_model)
# Add a judge: add a JudgeSpec entry to `JUDGES` in the cell above.

list_judge_ids()

['openai', 'claude_haiku', 'gemini_flash', 'mistral_medium']

In [107]:
run_judge("openai", df)

Output()

Criteria:
Vurder om den genererte oppsummeringen er korrekt i forhold til kildedokumentet (kontekst) og samsvarer godt med 
referanseoppsummeringen der det er relevant. 
 
Evaluation Steps:
[
    "Les gjennom kildedokumentet (kontekst) for å forstå hovedinnholdet.",
    "Sammenlign den genererte oppsummeringen (Actual Output) med kildedokumentet for å vurdere om den er korrekt og
dekker de viktigste punktene.",
    "Sammenlign den genererte oppsummeringen (Actual Output) med referanseoppsummeringen (Expected Output) for å 
vurdere samsvar og relevans.",
    "Vurder om Input (oppgavebeskrivelse) er fulgt i den genererte oppsummeringen."
] 
 
Rubric:
None 
 
Score: 0.08807970764133562
 
Reason: Den genererte oppsummeringen er kun en gjentakelse av oppgavebeskrivelsen og lister opp dokumentets 
metadata uten å oppsummere hovedinnholdet. Den dekker ikke de viktigste punktene fra kildedokumentet, som 
godkjenning av søknaden, tiltakets størrelse og formål, vurdering av miljøforhold, eller klagerett. Den samsvarer 
heller ikke med referanseoppsummeringen og følger ikke oppgavebeskrivelsen om å lage en faktisk oppsummering.

======================================================================

OpenAI (default) — score (0–1): 0.08807970764133562
Reason:
Den genererte oppsummeringen er kun en gjentakelse av oppgavebeskrivelsen og lister opp dokumentets metadata uten å oppsummere hovedinnholdet. Den dekker ikke de viktigste punktene fra kildedokumentet, som godkjenning av søknaden, tiltakets størrelse og formål, vurdering av miljøforhold, eller klagerett. Den samsvarer heller ikke med referanseoppsummeringen og følger ikke oppgavebeskrivelsen om å lage en faktisk oppsummering.


(0.08807970764133562, <deepeval.metrics.g_eval.g_eval.GEval at 0x149e48d60>)

In [13]:
import requests

url = "http://localhost:1234/api/v1/chat"

payload = {
    "model": "google/gemma-3-4b",
    "system_prompt": "You answer only in rhymes.",
    "input": "What is your favorite color?"
}

headers = {
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)

data = response.json()  # dict — subscript this, not `response` (a Response object)
print(data)
# Assistant text: data["output"][0]["content"]

{'model_instance_id': 'google/gemma-3-4b', 'output': [{'type': 'message', 'content': 'A sapphire hue, so deep and bright, \nA calming shade, a wondrous sight!'}], 'stats': {'input_tokens': 22, 'total_output_tokens': 20, 'reasoning_output_tokens': 0, 'tokens_per_second': 10.054510316536385, 'time_to_first_token_seconds': 10.849}, 'response_id': 'resp_07d1d0cfb605749ef0982cce5c4192c3080a92cdb8b4fce1'}


In [ ]:
data["output"][0]["content"]

In [15]:
type(response)

requests.models.Response

In [21]:
# G-Eval prompt + one eval row → local LLM (localhost:1234)
from pathlib import Path

import pandas as pd
import requests

ROOT = Path(".")
GEVAL_DIR = ROOT / "Data/prompts/geval"
# Pick one: faithfulness.txt | correctness.txt | completeness.txt
GEVAL_PROMPT_FILE = GEVAL_DIR / "faithfulness.txt"

EVAL_JSONL = ROOT / "Data/eval/norskgpt-llama3-8b-apptainer-checkpoint-5000-inputs-refs-preds-examples_1000.jsonl"
ROW_IDX = 0  # Summary A = reference; Summary B = prediction
DOC_MAX_CHARS = None  # e.g. 12_000 if local model context is too small

LOCAL_URL = "http://localhost:1234/api/v1/chat"
LOCAL_MODEL = "google/gemma-3-4b"

template = GEVAL_PROMPT_FILE.read_text(encoding="utf-8")
eval_df = pd.read_json(EVAL_JSONL, lines=True)
row = eval_df.iloc[ROW_IDX]

doc = str(row["input_text"])
if DOC_MAX_CHARS is not None and len(doc) > DOC_MAX_CHARS:
    doc = doc[:DOC_MAX_CHARS] + "\n\n[... document truncated ...]"

user_message = (
    template.replace("{{DOCUMENT}}", doc)
    .replace("{{SUMMARY_A}}", str(row["reference"]))
    .replace("{{SUMMARY_B}}", str(row["prediction"]))
)

system_prompt = (
    "You are an evaluator. Follow the user instructions exactly. "
    "If asked for one word (A, B, or Tie), reply with only that word."
)

payload = {
    "model": LOCAL_MODEL,
    "system_prompt": system_prompt,
    "input": user_message,
}
r = requests.post(
    LOCAL_URL,
    json=payload,
    headers={"Content-Type": "application/json"},
    timeout=300,
)
r.raise_for_status()
data = r.json()
print("Prompt file:", GEVAL_PROMPT_FILE.name)
print("Eval row:", ROW_IDX, "from", EVAL_JSONL.name)
print("--- assistant ---")
print(data["output"][0]["content"])



Prompt file: faithfulness.txt
Eval row: 0 from norskgpt-llama3-8b-apptainer-checkpoint-5000-inputs-refs-preds-examples_1000.jsonl
--- assistant ---
A
